# P78 — Una generalización decisional del aprendizaje en línea y su aplicación al boosting

## 1. Título y paper

**Paper:** *A Decision-Theoretic Generalization of On-Line Learning and an Application to Boosting*  
**Autoría:** Yoav Freund, Robert E. Schapire  
**Año y venue:** 1997 · Journal of Computer and System Sciences, 55(1), 119–139  
**Nivel:** L3 · **Motor:** `adaboost`  
**Ficha completa:** [`P78_adaboost`](../../papers/foundational/P78_adaboost/README.md)

**Hito:** Demuestra que muchos clasificadores apenas mejores que el azar se combinan en uno arbitrariamente bueno, y da el algoritmo que lo hace.

- [doi:10.1006/jcss.1997.1504](https://doi.org/10.1006/jcss.1997.1504)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Kearns y Valiant habían preguntado si un aprendiz «débil» —apenas mejor que el azar— puede convertirse en uno «fuerte». La respuesta afirmativa existía pero era impracticable: exigía conocer de antemano la ventaja del aprendiz débil.
2. Ejecutar una implementación mínima de la propuesta: AdaBoost: entrenar clasificadores en serie, subiendo el peso de los ejemplos que el anterior falló, y ponderar el voto de cada uno por su error. Se adapta solo a la calidad de cada aprendiz, sin conocerla de antemano.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- Schapire (1990), la fuerza del aprendizaje débil
- P74


## 4. Intuición

Un clasificador que apenas supera al azar parece inútil. Entrena uno, mira qué falla, sube el peso de esos ejemplos y entrena otro que se concentre en ellos. Repite. La suma ponderada de todos ellos resuelve lo que ninguno sabía resolver.


## 5. Concepto mínimo

```text
Para t = 1..T:
    hₜ ← aprendiz débil sobre la distribución de pesos Dₜ
    εₜ ← error PONDERADO de hₜ
    αₜ ← ½·ln((1 − εₜ)/εₜ)          ← más voto a quien menos falla
    Dₜ₊₁(i) ∝ Dₜ(i)·exp(−αₜ·yᵢ·hₜ(xᵢ))   ← sube el peso de lo fallado

H(x) = signo( Σ αₜ·hₜ(x) )
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('adaboost', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuánto acierta el mejor tocón individual sobre una banda central?
2. ¿Y el conjunto?
3. ¿Qué le pasa al peso del ejemplo más difícil?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('adaboost', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('adaboost', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Ningún tocón describe una banda: el mejor de los 22 acierta el **75 %**. El conjunto ponderado llega al **91,7 %** en la tercera ronda. Y el peso máximo pasa de 0,0833 —el reparto uniforme inicial— a 0,293: la atención se concentra donde el conjunto todavía falla.


## 10. Comentario pedagógico

La clave está en `α`: no es una media de opiniones, es una media **ponderada por competencia**, y el peso sale de una fórmula, no de un ajuste manual. Ese esquema —modelos en serie, cada uno corrigiendo el residuo del anterior— es el que domina hoy los datos tabulares en forma de gradient boosting y XGBoost.


## 11. Error o anti-patrón deliberado

Anti-patrón: aplicar boosting a datos con etiquetas ruidosas sin pensarlo.


In [ ]:
print('AdaBoost sube el peso de lo que falla. Si un ejemplo esta MAL etiquetado,')
print('nunca lo va a acertar, y su peso crece ronda tras ronda.')
print('El conjunto acaba dedicando su capacidad a memorizar el error.')

## 12. Corrección

Dónde mirar para detectarlo:


In [ ]:
r = run_paper_lab('adaboost', seed=7)['result']
print('peso maximo final:', max(r['pesos_finales']))
print('reparto inicial  :', round(1/r['ejemplos'], 4))
print('Un peso que se dispara sobre uno o dos ejemplos es una senal de alarma,')
print('no una senal de que el algoritmo esta trabajando bien.')

## 13. Desafío guiado

Sigue la historia del conjunto ronda a ronda y localiza en qué momento supera al mejor tocón individual.


In [ ]:
r = run_paper_lab('adaboost', seed=3)['result']
show(r)

## 14. Desafío autónomo

Aplica boosting a un conjunto real, mide exactitud en entrenamiento y en prueba a lo largo de las rondas, e identifica dónde empieza a sobreajustar. Después introduce un 5 % de etiquetas erróneas y repite.


## 15. Evidencia de aprendizaje

Guarda la historia del conjunto por rondas y tu explicación de por qué el ruido de etiqueta es el punto débil del método.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P78_adaboost/README.md) · evaluación formal: [`assessments/papers/P78_adaboost.md`](../../assessments/papers/P78_adaboost.md)


## 16. Cierre

Los modelos en serie funcionan. Falta la otra forma de combinar: en paralelo, y buscando que se parezcan lo menos posible.


## 17. Conexión con el siguiente hito

- P79

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
